Multilabel CNN for contraction number and duration 

In [2]:
import os.path as op
import mne 
import os
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score,ConfusionMatrixDisplay, f1_score, classification_report  # Import necessary metrics
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import matplotlib
import pickle

from sklearn.datasets import make_multilabel_classification
from sklearn.preprocessing import MultiLabelBinarizer

from scipy.fft import fft, ifft,fftfreq
from scipy.signal import welch, find_peaks 

from tensorflow.keras.models import Sequential, Model  # Import Sequential model from TensorFlow Keras
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Input, Dropout  # Import necessary layers from TensorFlow Keras
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2, l1

matplotlib.use('QtAgg') # for GUI 
mne.set_log_level("CRITICAL")

Define Model 

In [28]:
def CNN_model_twohead(input_shape, num_classes,feature_num):
    global epoch_len

    inputs = Input(shape=(epoch_len, feature_num))

    # Convolutional Layers - 
    x = Conv1D(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))(inputs) # Add a 1D convolutional layer with 32 filters and ReLU activation
    x = MaxPooling1D(pool_size=2)(x) # Add a max pooling layer

    x = Conv1D(64, kernel_size=3, activation='relu')(x)
    x = MaxPooling1D(pool_size=1)(x)

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    x = Flatten()(x)  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    
    count_output = Dense(num_classes, activation='softmax', name='count_output')(x)
    duration_output = Dense(1, activation='linear', name='duration_output')(x)

    model = Model(inputs=inputs, outputs=[count_output, duration_output])

    return model  # Return the compiled model

In [ ]:
def CNN_model_regression(input_shape, num_classes,feature_num):
    global epoch_len

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=1))  # Add another max pooling layer

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 

    model.add(Dense(2, activation='linear'))  # Add the output layer with linear to do regression 

    return model 

Loading in Data

In [20]:
features_all = pd.read_pickle("training_features_18042026.pkl")

# dummy column of duration 
features_all = features_all.assign(Duration_zygo = np.random.randint(0, 6, size=np.shape(features_all)[0]))
features_all = features_all.assign(Duration_corr = np.random.randint(0, 6, size=np.shape(features_all)[0]))


In [27]:
features_all.head()

,Subject,Nap Number,Triggers_Order_Nap,True_Muscle_Activated,Num_Contractions_Zygo,Num_Contractions_Corr,WL_Zygo,Var_Zygo,RMS_Zygo,MAVS_Zygo,WL_Corr,Var_Corr,RMS_Corr,MAVS_Corr,Zygo,Corr,Duration_zygo,Duration_corr
0,RL19RS,1,1,Corr,0,3,"[16.979501139871267, 18.165877955842884, 18.59...","[2.4357275001701533, 2.401023751914199, 2.3740...","[2.063547532557064, 2.0308865582680564, 1.9948...","[-0.037187916340846394, -0.04570457765223557, ...","[75.29678021851888, 76.39875013161338, 78.5484...","[3.0509425198291757, 3.047810217522341, 3.1377...","[1.7913263414331733, 1.7885089584884002, 1.803...","[-0.008847728745045247, 0.020195338894544834, ...","[-2.7277040853500525, 0.31356623348522716, 0.8...","[-0.7911568335676051, 1.912198159234712, -0.86...",1,5
1,RL19RS,1,2,Corr,0,3,"[26.824654542616372, 27.632174893961363, 27.76...","[0.8344136524953758, 0.8463100634295447, 0.862...","[0.9188357871709856, 0.927154584528815, 0.9381...","[0.016124978477620067, 0.018820054425093158, 0...","[88.84586351699936, 91.72494689471449, 94.9541...","[5.900218213130837, 6.0033229929637, 6.0065618...","[2.549990265745516, 2.551919890588013, 2.55310...","[0.003144624495356796, 0.0019711226458358766, ...","[-0.07304423025304638, -2.2181182276310256, -2...","[1.4867179586649941, 3.038561724391297, 0.4844...",4,4
2,RL19RS,1,3,Zygo,3,0,"[24.883849912053638, 25.700508948621653, 26.33...","[0.7468896743468332, 0.7497155323785327, 0.770...","[0.8650623074114399, 0.8669707886369807, 0.879...","[0.005861488405386139, 0.0186177487040049, 0.0...","[111.49377377841105, 112.30307576087087, 114.3...","[8.66934942006402, 8.668903914616124, 8.716578...","[2.9725897038242906, 2.970058549628821, 2.9811...","[-0.01646558841278223, 0.02292561840954077, 0....","[0.13543491172318878, -1.0559624223971724, -0....","[-0.8684031943153836, 0.22778864107415986, -0....",2,5
3,RL19RS,1,4,Zygo,3,0,"[24.477820981376315, 26.8812398019832, 27.3165...","[0.44507661010216365, 0.5037639088652561, 0.53...","[0.7626033518594519, 0.813616368572243, 0.8420...","[0.0283114732007127, 0.019604390522114823, 0.0...","[105.15945939835385, 113.62873732491099, 118.6...","[11.301421260842174, 11.568401993181608, 11.59...","[3.392698218078343, 3.4405619281245943, 3.4388...","[0.06157093211621323, -0.00626023389568342, 0....","[-0.7122686077788487, 0.11544473391500598, 0.2...","[1.1167298715401666, 1.5394483862679575, -0.22...",4,4
4,RL19RS,1,5,Corr,0,3,"[23.771646650420337, 25.250403446319936, 26.60...","[1.8706616369307074, 1.8540474174075106, 1.902...","[1.7844795652915169, 1.7573151489189478, 1.732...","[-0.03526536750283582, -0.027489667134452667, ...","[97.88042157234477, 107.45667002602518, 110.58...","[8.045422533890312, 8.250081346269646, 8.24998...","[2.8438684580292426, 2.88469073194194, 2.88459...","[0.061789720889836586, -0.00080370477739522, 0...","[-2.245731503659271, -1.7656176164993092, 0.67...","[-0.34758477119918696, -2.500891459504272, -0....",3,0


In [21]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y_contractions = np.concatenate([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])       

y_durations = np.concatenate([features_all_temp["Duration_zygo"].astype(int).to_numpy(),
                    features_all_temp["Duration_corr"].astype(int).to_numpy()])       

y = np.column_stack((y_contractions, y_durations))

indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [26]:
print(y[0])

[0 1]


In [ ]:
# Create CNN model using the adjusted input shape and number of classes
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 

cvScores_dur =[]
cvScores_contr =[]

epoch_num = 10 
k = 1
continuous = False 

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]

    model = CNN_model_twohead(input_shape, num_classes,feature_num) #CNN_model_regression if treating both as continuous

    if not continuous: # not treating as continuous 
        model.compile(optimizer='adam',loss=['sparse_categorical_crossentropy', 'mse'],metrics=['accuracy', 'mae'] ) #think about metric 
        model_history_kfold = model.fit(model.fit(X_train,[y_train[0], y_train[1]], 
                                              validation_data=(X_val,[y_val[0], y_val[1]]), 
                                              epochs=epoch_num))
        
        scores = model.evaluate(X_test,[y_test[0], y_test[1]])
        cvScores_dur.append(scores[1] * 100)
        cvScores_contr.append(scores[1] * 100)

    else:
        model.compile( optimizer='adam',loss='mse',metrics=['mae'])
        model_history_kfold = model.fit(X_train, y_train, epochs=epoch_num, validation_data=(X_val, y_val)) 

        scores = model.evaluate(X_test,y_test) # SEE WHAT SCORES PRINTS 
        cvScores_dur.append(scores[1] * 100)
        cvScores_contr.append(scores[1] * 100)

    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    #model_history_kfold = model.fit(X_train, y_train, epochs=epoch_num, validation_data=(X_val, y_val)) #, callbacks=[early_stop])
    
    #plot_accuracy(model_history_kfold,i)
    
    

    k += 1 
    

if continuous:
    model_history = model.fit(X_train_full, y_train_full, epochs=epoch_num, validation_data=(X_test, y_test)) #, callbacks=[early_stop])
else:
    model_history = model.fit(X_train_full, [y_train_full[0], y_train_full[1]], epochs=epoch_num, validation_data=(X_test, [y_test[0], y_test[1]])) #, callbacks=[early_stop])

In [ ]:
# save model 
with open("multilabel.pkl", "wb") as f:
    pickle.dump(model, f)

In [ ]:
# cross validation results 
avgScores_dur = np.mean(cvScores_dur)
stdScores_dur = np.std(cvScores_dur)

avgScores_contr= np.mean(cvScores_contr)
stdScores_contr = np.std(cvScores_contr)

print(f"Average KFold Cross Validation Score for contraction: {avgScores_contr}")
print(f"Standard Deviation KFold Cross Validation Score for contractionon: {stdScores_contr}")

print(f"Average KFold Cross Validation Score for duration: {avgScores_dur}")
print(f"Standard Deviation KFold Cross Validation Score for duration: {avgScores_dur}")

In [ ]:
# full training results (test data not seen during cross val)
[y_pred_train_contraction, y_pred_train_dur] = model.predict(X_train_full)  

y_pred_train_contraction = np.argmax(y_pred_train_contraction, axis=1)   
y_pred_train_dur = np.argmax(y_pred_train_dur, axis=1)   

# Predict on test data
[y_pred_test_contraction, y_pred_test_dur] = model.predict(X_test)  

y_pred_test_contraction = np.argmax(y_pred_test_contraction, axis=1)   
y_pred_test_dur = np.argmax(y_pred_test_dur, axis=1)   

# Calculate accuracy
accuracy_training_contraction = accuracy_score(y_train_full, y_pred_train_contraction)   
accuracy_training_dur = accuracy_score(y_train_full, y_pred_train_dur)   

accuracy_test_contraction = accuracy_score(y_test, y_pred_test_contraction)  
accuracy_test_dur= accuracy_score(y_test, y_pred_test_dur)  

# Calculate F1 score
f1_training_contraction = f1_score(y_train_full, y_pred_train_contraction, average='weighted')  
f1_training_dur = f1_score(y_train_full, y_pred_train_dur, average='weighted')  

f1_test_contraction = f1_score(y_test, y_pred_test_contraction, average='weighted')  
f1_test_dur = f1_score(y_test, y_pred_test_dur, average='weighted')  

# Print accuracy and F1 score
print("Training Accuracy :", accuracy_training_contraction)  
print("Test Accuracy :", accuracy_test_contraction)  
print("Training F1 Score :", f1_training_contraction)   
print("Test F1 Score :", f1_test_contraction)    
print('-----------------------------------------')
print("Training Accuracy :", accuracy_training_dur)
print("Test Accuracy :", accuracy_test_dur)
print("Training F1 Score :", f1_training_dur)
print("Test F1 Score :", f1_test_dur)